In [ ]:
# 사용자 요구 사항 및 스크린샷 지침을 따르는 딥러닝 실습 코드입니다.
# 사용자가 이전에 패키지 설치 시 문제를 겪었으므로, 코드를 실행하기 전에
# `uv pip install tensorflow matplotlib scikit-learn pandas`가
# 가상환경에 성공적으로 설치되었는지 확인해야 합니다.

import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

# --- [파트 1: 07-1 재현 - Fashion MNIST 단층 분류] ---
print("\n--- 파트 1: Fashion MNIST 단층 분류 실습 시작 ---")

# 데이터 로드
(fashion_train_full_input, fashion_train_full_target), (fashion_test_input, fashion_test_target) = keras.datasets.fashion_mnist.load_data()

# 지침 1: /255.0 정규화 적용 (데이터를 0~1 사이 값으로 변환)
# 원본 데이터는 0~255 사이의 픽셀 값입니다.
fashion_train_full_scaled = fashion_train_full_input / 255.0
fashion_test_scaled = fashion_test_input / 255.0

# 지침 2: reshape(-1, 784) 적용 (2차원 이미지를 1차원 벡터로 펼치기)
# (60000, 28, 28) 형태를 (60000, 784) 형태로 변경합니다.
fashion_train_full_flat = fashion_train_full_scaled.reshape(-1, 784)
fashion_test_flat = fashion_test_scaled.reshape(-1, 784)

# 지침 3: train_test_split(test_size=0.2, random_state=42) 적용 (검증 세트 분리)
# 60,000개의 데이터를 48,000개(학습)와 12,000개(검증)로 나눕니다.
fashion_train_input, fashion_val_input, fashion_train_target, fashion_val_target = train_test_split(
    fashion_train_full_flat, fashion_train_full_target, test_size=0.2, random_state=42
)

# 데이터 형태 및 분류 개수 확인
print(f"학습 데이터 형태: {fashion_train_input.shape}") # (48000, 784)
print(f"검증 데이터 형태: {fashion_val_input.shape}") # (12000, 784)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f"분류 개수: {len(class_names)} ({class_names})")

# 지침 4: Dense(10, activation='softmax') 모델 설계
# 입력 뉴런 784개, 출력 뉴런 10개(분류 개수)인 단층 신경망입니다.
# softmax 함수는 출력을 확률 값으로 변환합니다.
fashion_model = Sequential([
    # 입력 형태를 명시합니다. 딥러닝 빌딩의 기초 공사와 같습니다.
    Dense(10, activation='softmax', input_shape=(784,))
])

# 지침 5: sparse_categorical_crossentropy 손실 함수로 컴파일
# 타겟 값이 정수형(0~9)이므로 sparse 버전을 사용합니다.
fashion_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# 지침 07-1의 핵심: model.summary() 파라미터 수 검산
fashion_model.summary()
print("\n--- 파트 1: 파라미터 수 검산 설명 ---")
# 설명: 파라미터 수 = (입력 뉴런 수 × 출력 뉴런 수) + 출력 뉴런 수(편향)
# = (784 × 10) + 10 = 7840 + 10 = 7850개
# model.summary()의 Total params와 일치하는지 확인합니다.

# 학습 (코드 재현이 목적이므로 간략하게 진행)
print("\n--- 파트 1: 학습 진행 ---")
fashion_history = fashion_model.fit(fashion_train_input, fashion_train_target, epochs=5,
                                  validation_data=(fashion_val_input, fashion_val_target))

# 성능 확인
print("\n--- 파트 1: 성능 결과 ---")
val_accuracy = fashion_history.history['val_accuracy'][-1]
print(f"검증 세트 정확도: {val_accuracy:.4f}")


# --- [파트 2: Iris MLP - 단층 vs 2층 성능 비교] ---
print("\n--- 파트 2: Iris MLP 성능 비교 실습 시작 ---")

# 데이터 로드 (Keras가 아닌 Pandas로 간편하게 로드)
iris_url = "https://raw.githubusercontent.com/rickiepark/hg-mldl/master/iris.csv"
iris_df = pd.read_csv(iris_url)

# 특성(X)과 타겟(y) 분리
iris_X = iris_df.iloc[:, :-1].values # 꽃받침/꽃잎 길이/너비 4개 특성
iris_y_raw = iris_df.iloc[:, -1].values # 꽃 종류 (Setosa, Versicolor, Virginica)

# 타겟 데이터 정수형으로 변환 (Keras 분류용)
# sklearn의 LabelEncoder를 사용해도 되지만, 간편하게 매핑합니다.
iris_y = np.where(iris_y_raw == 'Setosa', 0,
                 np.where(iris_y_raw == 'Versicolor', 1, 2))

# 학습 및 테스트 세트 분리 (iris는 데이터가 적으므로 80:20 비율 사용)
iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris_X, iris_y, test_size=0.2, random_state=42
)

# 지침 1: StandardScaler 표준화 적용
# 각 특성의 평균을 0, 분산을 1로 맞춥니다.
iris_scaler = StandardScaler()
iris_X_train_scaled = iris_scaler.fit_transform(iris_X_train)
iris_X_test_scaled = iris_scaler.transform(iris_X_test)

# 지침 2: 단층: Dense(3, 'softmax') 모델 설계
# 입력 뉴런 4개, 출력 뉴런 3개인 단층 신경망입니다.
print("\n--- 파트 2: 단층 모델 설계 및 학습 ---")
iris_model_1 = Sequential([
    Dense(3, activation='softmax', input_shape=(4,))
])

iris_model_1.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

iris_model_1.summary()
print("\n--- 파트 2: 단층 모델 파라미터 수 검산 ---")
# 설명: (4 × 3) + 3 = 12 + 3 = 15개

# 지침 4: epochs=200으로 충분히 학습
iris_history_1 = iris_model_1.fit(iris_X_train_scaled, iris_y_train, epochs=200, verbose=0,
                                 validation_data=(iris_X_test_scaled, iris_y_test))

# 지침 3: 2층: Dense(8, 'sigmoid') -> Dense(3, 'softmax') 모델 설계
# 은닉층(8개 뉴런, sigmoid)과 출력층(3개 뉴런, softmax)을 가진 2층 신경망입니다.
print("\n--- 파트 2: 2층 모델 설계 및 학습 ---")
iris_model_2 = Sequential([
    Dense(8, activation='sigmoid', input_shape=(4,)),
    Dense(3, activation='softmax')
])

iris_model_2.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

iris_model_2.summary()
print("\n--- 파트 2: 2층 모델 파라미터 수 검산 ---")
# 설명:
# 은닉층: (4 × 8) + 8 = 32 + 8 = 40개
# 출력층: (8 × 3) + 3 = 24 + 3 = 27개
# 총 파라미터: 40 + 27 = 67개

# 지침 4: epochs=200으로 충분히 학습
iris_history_2 = iris_model_2.fit(iris_X_train_scaled, iris_y_train, epochs=200, verbose=0,
                                 validation_data=(iris_X_test_scaled, iris_y_test))

# 지침: 성능 차이와 파라미터 수 차이 비교 해석
print("\n--- 파트 2: 성능 및 파라미터 비교 결과 ---")

# 최종 성능 확인
print("단층 모델:")
loss1, acc1 = iris_model_1.evaluate(iris_X_test_scaled, iris_y_test, verbose=0)
print(f"  테스트 세트 정확도: {acc1:.4f}")
print(f"  총 파라미터 수: {iris_model_1.count_params()}개")

print("2층 모델:")
loss2, acc2 = iris_model_2.evaluate(iris_X_test_scaled, iris_y_test, verbose=0)
print(f"  테스트 세트 정확도: {acc2:.4f}")
print(f"  총 파라미터 수: {iris_model_2.count_params()}개")

print("\n--- 파트 2: 비교 해석 ---")
# 해석:
# 2층 모델은 은닉층을 추가하여 더 복잡한 특징을 학습할 수 있으므로,
# 일반적으로 단층 모델보다 더 높은 정확도를 보입니다.
# 하지만 파라미터 수(67개)가 단층 모델(15개)보다 약 4배 이상 많으므로,
# 모델 크기가 커지고 학습 시간이 약간 더 오래 걸립니다.
# Iris 데이터셋은 매우 단순하므로 정확도 차이가 크지 않을 수 있지만,
# 복잡한 데이터셋에서는 은닉층을 더 깊게 쌓는 것이 성능 향상의 핵심입니다.